[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/lakelogic/blob/main/examples/colab/07_lifecycle.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/LakeLogic/lakelogic/blob/main/examples/colab/07_lifecycle.ipynb)

> Replace `LakeLogic/lakelogic` + `main` in the badge links when this repo is published.


# 07 - Lifecycle: cloud in, nested out, cloud back out

Three recipes that fill the gaps between the existing gallery notebooks - the
parts of the ingestion/lifecycle story you'd otherwise hunt for across pages:

1. **Cloud object-store ingestion (Bronze)** - one contract reads S3 *or* GCS *or* ADLS. Cloud is just a URI prefix.
2. **Semi-structured / nested JSON (Bronze -> Silver)** - flatten nested objects into typed, flat Silver columns.
3. **Export / egress (Gold -> cloud bucket)** - publish validated Gold data back out to an object store.

Same philosophy as the rest of the gallery: **lean contracts, one obvious path,
try any pattern in ~30 seconds.** Each cell writes a tiny contract + a few rows of
stand-in data, then runs it with `lakelogic run` so it works on Colab with **zero
cloud credentials**. Where a step would need real storage, we point the run at a
local stand-in and show the cloud URI in the contract.


In [ ]:
# Install LakeLogic (polars + duckdb engines are enough for these recipes)
!pip install -q lakelogic[polars,duckdb]

from pathlib import Path
DEMO = Path('07_lifecycle_demo')
DEMO.mkdir(exist_ok=True)
print('LakeLogic installed. Writing recipes into', DEMO.resolve())


## Recipe 1 - Cloud object-store ingestion (Bronze), one contract for ANY cloud

The `source.path` is a **cloud URI**. Swap the scheme and the *same contract*
reads from AWS S3, Google GCS, or Azure ADLS - LakeLogic resolves the storage
backend from the URI and picks up credentials from the standard environment
variables (`AWS_*`, `GOOGLE_APPLICATION_CREDENTIALS`, `AZURE_*`).

```
s3://lakelogic-demo/orders/        # AWS S3
gs://lakelogic-demo/orders/        # Google GCS
abfss://orders@acct.dfs.core.windows.net/   # Azure ADLS
```

> To run against real cloud storage: set your creds and drop the `--source`
> override so the run uses the contract's cloud `path`.


In [ ]:
cloud_orders_yaml = '''version: 1.0.0
dataset: cloud_orders
info:
  title: bronze_cloud_orders
  owner: data-team@company.com
  target_layer: bronze
source:
  type: landing
  path: s3://lakelogic-demo/orders/   # swap gs:// or abfss:// - same contract, cloud is just a prefix
  format: csv
model:
  fields:
    - {name: order_id, type: integer, required: true}
    - {name: customer_email, type: string, required: true, pii: true, masking: partial}
    - {name: amount, type: float, required: true}
    - {name: currency, type: string}
    - {name: status, type: string}
    - {name: created_at, type: string}
quality:
  row_rules:
    - {name: positive_amount, sql: "amount > 0"}
    - {name: valid_currency, sql: "currency IN ('GBP','USD','EUR')"}
'''
(DEMO / 'cloud_orders.yaml').write_text(cloud_orders_yaml)

# A tiny LOCAL stand-in for the cloud bucket so this runs with zero creds.
orders_csv = '''order_id,customer_email,amount,currency,status,created_at
1001,ada@example.com,42.50,USD,shipped,2026-02-01T09:14:00Z
1002,grace@example.com,19.99,EUR,pending,2026-02-01T10:02:00Z
1003,alan@example.com,88.00,GBP,delivered,2026-02-01T11:20:00Z
1004,linus@example.com,7.25,USD,returned,2026-02-02T08:41:00Z
1005,margaret@example.com,120.00,GBP,shipped,2026-02-02T12:00:00Z
'''
(DEMO / 'orders_sample.csv').write_text(orders_csv)

# Point --source at the local stand-in; the contract still declares the s3:// URI.
!lakelogic run -c 07_lifecycle_demo/cloud_orders.yaml -s 07_lifecycle_demo/orders_sample.csv


## Recipe 2 - Semi-structured / nested JSON (Bronze -> Silver)

The source is JSON with a **nested object** (`customer.address`). Setting
`source.flatten_nested: true` expands every nested object into flat
`parent_child` columns (recursively, up to 5 levels) *before* schema validation:

```
customer.id              -> customer_id
customer.name            -> customer_name
customer.address.city    -> customer_address_city
customer.address.country -> customer_address_country
```

So the Silver `model.fields` are the **flattened, typed names**. A scalar array
(`tags`) has no child keys to promote, so it's left as-is and dropped from the
flat Silver projection by `schema_policy.unknown_fields: drop`. To flatten only
specific columns, pass a list: `flatten_nested: [customer]`.


In [ ]:
nested_yaml = '''version: 1.0.0
dataset: nested_events
info:
  title: silver_nested_events
  owner: data-team@company.com
  target_layer: silver
source:
  type: landing
  path: ./07_lifecycle_demo/nested_events.json
  format: json
  flatten_nested: true
model:
  fields:
    - {name: event_id, type: integer, required: true}
    - {name: event_type, type: string, required: true}
    - {name: customer_id, type: integer, required: true}
    - {name: customer_name, type: string}
    - {name: customer_address_city, type: string}
    - {name: customer_address_country, type: string}
quality:
  row_rules:
    - {name: known_event_type, sql: "event_type IN ('signup','purchase')"}
    - {name: valid_country, sql: "customer_address_country IN ('GB','DE','US','FR')"}
server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"
'''
(DEMO / 'nested_events.yaml').write_text(nested_yaml)

nested_json = '''[
  {"event_id": 1, "event_type": "signup",   "customer": {"id": 42, "name": "Ada",   "address": {"city": "London",     "country": "GB"}}, "tags": ["beta", "eu"]},
  {"event_id": 2, "event_type": "purchase", "customer": {"id": 43, "name": "Grace", "address": {"city": "Berlin",     "country": "DE"}}, "tags": ["vip"]},
  {"event_id": 3, "event_type": "signup",   "customer": {"id": 44, "name": "Alan",  "address": {"city": "Manchester", "country": "GB"}}, "tags": []}
]'''
(DEMO / 'nested_events.json').write_text(nested_json)

!lakelogic run -c 07_lifecycle_demo/nested_events.yaml -s 07_lifecycle_demo/nested_events.json --output-good 07_lifecycle_demo/nested_silver.csv
print('--- flattened Silver output ---')
print((DEMO / 'nested_silver.csv').read_text())


## Recipe 3 - Export / egress (Gold -> cloud object store)

The final lifecycle stage: publish validated **Gold** data *out* to a cloud
bucket for downstream consumers. LakeLogic materializes to the same cloud URIs
it reads from (parquet / csv / delta), resolving the backend from the URI scheme.
Egress is just `materialize` with a cloud `target_path`:

```
s3://lakelogic-exports/gold/orders/
gs://lakelogic-exports/gold/orders/
abfss://exports@acct.dfs.core.windows.net/gold/orders/
```

For this runnable cell we override the destination with a **local path** via
`--materialize-target`. Set your storage creds and drop the override to egress
to the real bucket in the contract.

> **SFTP egress** is not yet a wired sink (`SFTPConnector` is ingress-only) - see
> the **Roadmap** row in `examples/README.md`.


In [ ]:
gold_yaml = '''version: 1.0.0
dataset: gold_orders_export
info:
  title: gold_orders_export
  owner: data-team@company.com
  target_layer: gold
model:
  fields:
    - {name: order_id, type: integer, required: true}
    - {name: amount, type: float, required: true}
    - {name: currency, type: string}
    - {name: status, type: string}
quality:
  row_rules:
    - {name: positive_amount, sql: "amount > 0"}
materialization:
  target_path: s3://lakelogic-exports/gold/orders/   # swap gs:// or abfss:// - cloud is just a prefix
  format: parquet
  strategy: overwrite
'''
(DEMO / 'gold_orders_export.yaml').write_text(gold_yaml)

# Reuse the orders stand-in from Recipe 1; override the cloud target with a local path.
!lakelogic run -c 07_lifecycle_demo/gold_orders_export.yaml -s 07_lifecycle_demo/orders_sample.csv --materialize --materialize-target 07_lifecycle_demo/gold_export.parquet

import polars as pl
print('--- Gold parquet written locally (would be s3:// with creds) ---')
print(pl.read_parquet('07_lifecycle_demo/gold_export.parquet'))


## Where next

- **Full gallery index:** [`examples/README.md`](https://github.com/LakeLogic/lakelogic/blob/main/examples/README.md) - every example organized by lifecycle stage, ingestion source, and data shape.
- **All contract options in one place:** the annotated reference contracts in `examples/reference/` (`bronze.annotated.yaml`, `silver.annotated.yaml`, `gold.annotated.yaml`).
- **Integrations** (`06_integrations.ipynb`): dbt, dlt/API, database, CDC, streaming.
- **Data mesh at scale** (`04_lakehouse_data_platform`): domain ownership + engine portability.
